# Writing a Data Product with FORM

This exercise starts with:
- a toy physics algorithm: `simulate_wires`
- a toy data product produced by that algorithm: `wire_readout`
- phlex components to run that algorithm: `simulate_wires_provider` and `write_wires.jsonnet`
- a script to check the results: `plot_one_wire.py`
- a CMake build system to put it all together

When you complete this exercise, you will be able to:
- configure phlex to write data products using FORM
- write data products to a file that FORM can read
- demonstrate that the data products in this FORM file contain plausible information

This notebook follows `README.md` in this directory step by step. Shell commands run in code cells
that start with `!` or `%%bash`. When a step asks you to **edit a file**, open it from the JupyterLab
file browser on the left, make the change, and save it (Ctrl+S) before running the next cell.

# Section 0: Software Setup

You need an environment that has **phlex** (with FORM), **ROOT** (with PyROOT) and **CMake 3.31 or later**.
On the DUNE Tech JupyterHub these come from the phaindia apptainer image.

**Choose the phaindia apptainer kernel** from the kernel menu at the top right of this notebook
(or **Kernel → Change Kernel…**). The kernel runs inside the image, so every `!` and `%%bash` cell
below can see phlex.

**Optional:** if you want to run commands by hand, open a terminal (File → New → Terminal) and launch
the same image there. You don't need this for the notebook.

In [ ]:
# From a terminal, to move to the ${HOME}/form-tutorial-2026 directory, run:
cd ${HOME}/form-tutorial-2026

# Then, from the same terminal, to launch the container, run:
apptainer shell --cleanenv /opt/images/phaindia-0.1-20261002.sif

Run the next cell to check that the tools are visible to this notebook. Run it again if you restart the kernel.

In [ ]:
import os
import subprocess
import sys

!which phlex cmake root python python3
!cmake --version | head -1
print("kernel python:", sys.executable)

# The kernel does not put PyROOT on the Python path, so add ROOT's library directory here.
# PYTHONPATH covers scripts that this notebook runs, such as plot_one_wire.py.
ROOT_LIBDIR = subprocess.check_output(["root-config", "--libdir"], text=True).strip()
if ROOT_LIBDIR not in sys.path:
    sys.path.insert(0, ROOT_LIBDIR)
if ROOT_LIBDIR not in os.environ.get("PYTHONPATH", "").split(":"):
    os.environ["PYTHONPATH"] = f"{ROOT_LIBDIR}:{os.environ.get('PYTHONPATH', '')}"

import ROOT
print("PyROOT", ROOT.gROOT.GetVersion())

## Paths and environment

The tutorial installs into `opt/` inside the repository and builds in `opt/build`.
The next cell sets those paths and does the same job as `setup.sh` (which the build installs into
`opt/bin`), so that `phlex` can find the plugins you build.

Run this cell again if you restart the kernel.

In [ ]:
import os
from pathlib import Path

WRITING_DIR = Path.cwd()
assert (WRITING_DIR / "write_wires.jsonnet").exists(), "Open this notebook from the writing/ directory"

SOURCE_DIR = WRITING_DIR.parent
PREFIX = SOURCE_DIR / "opt"
BUILD_DIR = PREFIX / "build"

os.environ["SOURCE_DIR"] = str(SOURCE_DIR)
os.environ["PREFIX"] = str(PREFIX)
os.environ["BUILD_DIR"] = str(BUILD_DIR)

# Same as setup.sh.in
for var, sub in [("LD_LIBRARY_PATH", "lib"), ("PATH", "bin"), ("PHLEX_PLUGIN_PATH", "lib")]:
    os.environ[var] = f"{PREFIX / sub}:{os.environ.get(var, '')}"

print("source: ", SOURCE_DIR)
print("build:  ", BUILD_DIR)
print("install:", PREFIX)

## Build the tutorial

Configure and build the tutorial once before you start. You will rebuild it later in the exercise.

This cell deletes `opt/build` first, so running it again always gives a clean build.

In [ ]:
%%bash
rm -rf "$BUILD_DIR"
mkdir -p "$BUILD_DIR"
cd "$BUILD_DIR"
cmake "$SOURCE_DIR" -DCMAKE_INSTALL_PREFIX="$PREFIX" -DCMAKE_BUILD_TYPE=Release
make -j4 install

# Section 1: Run phlex without FORM

**Try running `write_wires.jsonnet` now.** The job should complete, but it won't produce an output file.

In [ ]:
!phlex -c write_wires.jsonnet -j 1 --stage sim
!ls -l *.root 2>/dev/null || echo "No .root file yet, as expected."

# Section 2: Configuring FORM

`write_wires.jsonnet` drives phlex to run the `simulate_wires` algorithm. But it doesn't save results
to a file. It can only pass data products to other algorithms in the same process.

Use FORM by scheduling a special algorithm in a phlex configuration file like this:
```
output: {
  cpp: 'form_module',
  outputs: {
    sim_wires: {
      output_file: 'form-tutorial-writing.root',
      technology: 'ROOT_RNTUPLE',
      products: ['sim',],
    },
  },
}
```

**Add this code block to `write_wires.jsonnet` now.** It needs to be at the same scope as the
`count_wires` algorithm.

The `output` and `cpp: form_module` parts will always stay the same. `sim_wires` defines a group of
data products that phlex will write. The name `sim_wires` does not need to match any other names and is
only present for user convenience. Each group of data products is written to the same `output_file`.
`technology` is the format in which those data products will be written. The default is ROOT's `TTree`
format. This tutorial focuses on the new `RNTuple` format because it produces smaller files.

`products` is a list of phlex **suffixes** whose data products will be written to this file. The
suffixes FORM is configured for match the second argument to `output_product()` on line 41 of
`simulate_wires_provider.cpp`. Suffixes look a little different for most other
kinds of phlex higher order nodes. The writing exercise in this tutorial will demonstrate how to save
data products from a transform node.

Check your edit:

In [ ]:
!cat write_wires.jsonnet

**Run `write_wires.jsonnet` now.** You should get output like this:
```
=== form_output_module::save_data_products ===
Creator: wires_provider:simulated_wires
Stage: sim
Data cell: [spill:0, apa:0]
Number of products: 1
  Product: simulate_wires/sim
root_rfield_write_container_imp::setup_write could not create column-wise storage for std::vector<raw::wire_readout, std::allocator<raw::wire_readout> >.  This class is probably using something obsolete like TLorentzVector.  Storing it in streamer mode to keep the application going.
Wrote 1 products to FORM
Simulated 1024 wires.

=== form_output_module::save_data_products ===
Creator: wires_provider:simulated_wires
Stage: sim
Data cell: [spill:0, apa:1]
Number of products: 1
  Product: simulate_wires/sim
Wrote 1 products to FORM
Simulated 1024 wires.
```

In [ ]:
!phlex -c write_wires.jsonnet -j 1 --stage sim

FORM is telling you that you're missing some ingredients for producing a readable file. You can open
`form-tutorial-writing.root` and see that it contains something. (In a terminal you would use
`root -l form-tutorial-writing.root` and `.ls`. The next cell does the same from Python.) But
`plot_one_wire.py` won't work yet, and the reading tutorial would crash if you ran it now. You're missing
a ROOT dictionary that tells FORM how to put data products back together from the contents of this file.

In [ ]:
import ROOT

f = ROOT.TFile.Open("form-tutorial-writing.root")
f.ls()
f.Close()

# Section 3: ROOT Dictionary

When you use the `ROOT_TTree` or `ROOT_RNTuple` FORM technologies, you need to produce a ROOT dictionary
for each data product that will be written to a file.

Most of ROOT dictionary generation happens when you compile your data product into a library. This
tutorial uses CMake to compile and link C++ libraries. Open `CMakeLists.txt` in this directory and
**add these code blocks now after `install(TARGETS wire_readout DESTINATION lib)`:**
```
reflex_generate_dictionary(wire_readout dict.h SELECTION classes_def.xml)
target_link_libraries(wire_readout PRIVATE ROOT::RIO)
set_target_properties(wire_readout PROPERTIES LIBRARY_OUTPUT_DIRECTORY ${CMAKE_CURRENT_BINARY_DIR})
install(FILES ${CMAKE_CURRENT_BINARY_DIR}/wire_readout_rdict.pcm DESTINATION lib)
```

Check your edit:

In [ ]:
!cat CMakeLists.txt

**Re-compile the tutorial** with the next cell. You'll have to do this each time you edit a file.
The jsonnet file is an exception to this rule.

**Run `write_wires.jsonnet` again.** The build system will tell you you're missing a few more files:
- `classes_def.xml`: tells ROOT which classes to build dictionaries for
- `dict.h`: headers for the classes in `classes_def.xml`

In [ ]:
%%bash
cd "$BUILD_DIR"
make -j4 install || echo "Build failed, as expected: the error above names a missing file."

**Create files that look like these.** Running each cell below writes the file into this directory.

In [ ]:
%%writefile classes_def.xml
<lcgdict>
  <class name="raw::wire_readout"/>
  <class name="std::vector<raw::wire_readout>"/>
</lcgdict>

In [ ]:
%%writefile dict.h
#include "wire_readout.hpp"
#include <vector>

Once you add these two files, you'll need to **re-create the build directory**:

In [ ]:
%%bash
rm -rf "$BUILD_DIR"
mkdir -p "$BUILD_DIR"
cd "$BUILD_DIR"
cmake "$SOURCE_DIR" -DCMAKE_INSTALL_PREFIX="$PREFIX" -DCMAKE_BUILD_TYPE=Release
make -j4 install

Now **run `write_wires.jsonnet` again.** The warning about streamer mode should be gone. This should be
a well-formed FORM file! The reading tutorial will demonstrate how to use it.

In [ ]:
!phlex -c write_wires.jsonnet -j 1 --stage sim

# Section 4: Check the Results

This tutorial will use some expert knowledge of how FORM works to check that the results make sense.
`plot_one_wire.py` draws only one of the simulated wires as a histogram that kind of looks like the old
LArSoft event display. Its arguments are the file name, the spill, the APA and the channel.

**Run `python plot_one_wire.py form-tutorial-writing.root 0 13 243`.** This should produce an image file
named `waveform_spill_0_apa_13.png`. (`{sys.executable}` runs the script with the same Python as this
notebook, the one that has PyROOT.)

In [ ]:
!{sys.executable} plot_one_wire.py form-tutorial-writing.root 0 13 243

Verify that it looks like a noise background with maybe a handful of Gaussian hits superimposed on top.

In [ ]:
from IPython.display import Image

Image("waveform_spill_0_apa_13.png")

Try a few other spills, APAs and channels by changing the arguments above. This job made 1 spill with
24 APAs (numbered 0 to 23) and 1024 channels per APA (numbered 0 to 1023).

# Section 5 (optional): Things to try

These are not in the README. Each one changes one thing and asks you to compare. **Keep
`form-tutorial-writing.root` from Section 3 as it is**, because the reading exercise reads it: give
every new run its own `output_file`.

**Task 1: look around the file (about 10 minutes).** Use the plot cell above to draw at least three
wires. Find one with a large signal pulse and one with only noise. Do APA 0 and APA 23 look alike?
Every wire uses the same ten noise frequencies, so the noise looks alike everywhere, while the
pulses land at random.

**Task 2: change the simulation (about 20 minutes).** `simulate_wires_provider` reads optional settings
from its block in `sources` (see `PHLEX_REGISTER_PROVIDERS` in `simulate_wires_provider.cpp`): `seed`,
`n_channels_per_apa`, `n_time_ticks`, `n_noise_functions` and `background_height`.
- (a) Copy `write_wires.jsonnet` to `write_wires_seed.jsonnet`, add `seed: 42,` to the `wires_provider`
  block and set `output_file` to `form-tutorial-seed.root`. Run it twice and plot the same wire each
  time. Then remove `seed` and do it again. What changes? The seed defaults to the clock, which is
  why your plots differ from your neighbour's.
- (b) Set `background_height: 10` (the default is 5) and plot the same wire. Look at the y-axis. The
  signal heights in `simulate_wires.cpp` are scaled with the noise, so the picture keeps its shape.

**Task 3: TTree or RNTuple (about 10 minutes).** Copy `write_wires.jsonnet` to
`write_wires_ttree.jsonnet`, change `technology` to `'ROOT_TTREE'` and `output_file` to
`form-tutorial-ttree.root`, and run it. Compare the two file sizes with `ls -l *.root`. Which is
smaller? (`plot_one_wire.py` reads RNTuples only, so it cannot open the TTree file.)

**Task 4 (stretch, about 25 minutes): every ADC value in one APA.** `plot_one_wire.py` reads one wire
with an `RNTupleReader` view; entry number = channel + 1024 × APA (for spill 0). Loop over the 1024
wires of one APA and fill one histogram with every ADC value, 0 to 300 ADC. The noise makes a bell
shape between 0 and 100 and the signal pulses make a tail. Keep this plot in mind for the reading
exercise, where a hit finder cuts at 60 ADC.

**Next:** the reading exercise uses `form-tutorial-writing.root` as its input.